# Fig. 4a — Metacluster-marker staircase dot plot for VGN subtypes

Four genes (`Lypd1`, `Sall3`, `Vcan`, `Lmo3`) define the original 4-metacluster (4-MC) scheme for VGNs. The plot is a **staircase / diagonal dot plot**: cluster rows are ordered so each marker dominates a contiguous block of subtypes, exposing the molecular structure of the 4-MC partition.

> Note: the downstream cNMF / GO enrichment analysis in Fig 4c uses the consolidated **3-MC** scheme (Lypd1+ / Vcan+ / Lmo3+), where the original `Sall3+` MC was redistributed into the other three (Calb2 → Lypd1+, Aldh1a3 → Vcan+, Nxph4 → Lmo3+). Fig 4a deliberately keeps `Sall3` as a 4th panel column because Sall3 expression alone still cleanly marks the calyx-Type-I population at the gene level — it is informative even though Sall3+ is no longer kept as a separate MC.

**Input**
- `adata_neuron_0310.h5ad` — VGN AnnData (chosen over 0306 because all 4 genes incl. `Vcan` are present in `var_names`)

**Outputs**
- `figures/Fig4a_metacluster_staircase_dotplot.pdf` (+ `.svg`, `.png`)
- `figures/Fig4a_metacluster_staircase_dotplot_values.csv` — underlying mean / fraction matrix
- `figures/Fig4a_metacluster_staircase_dotplot_cluster_order.csv` — final row order with metadata

**Plot conventions**
- Dot **size** = fraction of cells with detection (using `umi` layer for binarisation).
- Dot **colour** = mean log-normalised expression (`log_norm` layer), then **min-max scaled to [0, 1] per gene** (`var`-style scaling), so each column is independently normalised — within-gene contrast, not absolute magnitude.
- Cluster row order is **hand-curated** (`MANUAL_CLUSTER_ORDER`) to produce the staircase pattern; reorder by editing `MANUAL_CLUSTER_ORDER` and `GENES`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from scipy import sparse

plt.style.use('default')
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 7,
    'axes.titlesize': 8,
    'axes.labelsize': 7,
    'xtick.labelsize': 6,
    'ytick.labelsize': 8,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

DATA_DIR = Path('..')
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)

## 1. Parameters

In [ ]:
ADATA_PATH = DATA_DIR / 'adata_neuron_0310.h5ad'
GROUPBY = 'all_cluster_gene_name'

# Four-marker panel — Vcan placed before Lmo3 so the staircase aligns with the
# Vcan-dominant block coming before the Lmo3-dominant block in cluster order.
GENES = ['Lypd1', 'Sall3', 'Vcan', 'Lmo3']

# Hand-curated cluster order — edit here if cluster naming or composition changes
MANUAL_CLUSTER_ORDER = [
    'neuron_Vglut1/Calb1',
    'neuron_Gstm2',
    'neuron_Meis2',
    'neuron_Vglut1/Lypd1',
    'neuron_Vglut1/Calb2',
    'neuron_Nxph4',
    'neuron_Vglut1/Aldh1a3',
    'neuron_Slc2a4',
    'neuron_Vglut1/Nkain3',
    'neuron_Gsg1l',
    'neuron_Ptgfr',
    'neuron_Lmo3',
    'neuron_Fxyd7',
]

# Figure style
ORIENTATION = 'vertical'         # 'vertical' = genes on x-axis, clusters on y-axis
FIG_WIDTH, FIG_HEIGHT = 3.5, 5.2
DOT_SIZE_MIN = 20
DOT_SIZE_MAX_ADD = 3.0           # dot area = DOT_SIZE_MIN + pct_expr * DOT_SIZE_MAX_ADD
X_LABEL_ROTATION = 0
SHORTEN_CLUSTER_LABELS = True    # strip 'neuron_' prefix for compact axis labels
AXIS_PAD = 0.7                   # extra padding so edge dots don't clip tick labels

OUT_PREFIX = FIG_DIR / 'Fig4a_metacluster_staircase_dotplot'

cmap_wdg = mcolors.LinearSegmentedColormap.from_list(
    'white_darkgrey', cm.Greys(np.linspace(0.0, 0.83, 256))
)

## 2. Load adata and verify gene + cluster availability

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)

missing_genes = [g for g in GENES if g not in adata.var_names]
if missing_genes:
    raise ValueError(f'Missing genes in {ADATA_PATH}: {missing_genes}')
if GROUPBY not in adata.obs:
    raise ValueError(f'Missing obs column: {GROUPBY}')

if not hasattr(adata.obs[GROUPBY].dtype, 'categories'):
    adata.obs[GROUPBY] = adata.obs[GROUPBY].astype('category')

present = list(adata.obs[GROUPBY].cat.categories)
missing_manual = [c for c in MANUAL_CLUSTER_ORDER if c not in present]
extras = [c for c in present if c not in MANUAL_CLUSTER_ORDER]
if missing_manual:
    raise ValueError(f'Manual order contains clusters not in adata: {missing_manual}')
if extras:
    print(f'NOTE: clusters not in MANUAL_CLUSTER_ORDER (appended at end): {extras}')

cluster_order = MANUAL_CLUSTER_ORDER + extras
print(f'Genes:    {GENES}')
print(f'Clusters: {len(cluster_order)} (manual + {len(extras)} extras)')
print(f'Layers:   {list(adata.layers.keys())}')

## 3. Compute per-(cluster, gene) mean expression and detection fraction

In [ ]:
def to_dense_1d(x):
    return x.toarray().ravel() if sparse.issparse(x) else np.asarray(x).ravel()

def minmax_by_gene(df):
    """Min-max scale `mean_log_norm` to [0, 1] within each gene (column)."""
    parts = []
    for _, sub in df.groupby('gene', sort=False):
        vals = sub['mean_log_norm'].to_numpy(dtype=float)
        lo, hi = vals.min(), vals.max()
        scaled = np.zeros_like(vals) if np.isclose(hi, lo) else (vals - lo) / (hi - lo)
        tmp = sub.copy()
        tmp['mean_scaled_0_1'] = scaled
        parts.append(tmp)
    return pd.concat(parts, ignore_index=True)

# colour = mean of log_norm, size = % cells with umi > 0
expr_layer = adata.layers['log_norm'] if 'log_norm' in adata.layers else adata.X
detect_layer = adata.layers['umi'] if 'umi' in adata.layers else expr_layer
group_values = adata.obs[GROUPBY].astype(str).to_numpy()

rows = []
for cluster in present:
    mask = group_values == cluster
    for gene in GENES:
        gi = adata.var_names.get_loc(gene)
        expr = to_dense_1d(expr_layer[mask, gi])
        detect = to_dense_1d(detect_layer[mask, gi])
        rows.append({
            'cluster': cluster, 'gene': gene,
            'n_cells': int(mask.sum()),
            'mean_log_norm': float(np.mean(expr)),
            'pct_expr': float(np.mean(detect > 0) * 100.0),
        })

plot_df = minmax_by_gene(pd.DataFrame(rows))
plot_df['cluster'] = pd.Categorical(plot_df['cluster'], categories=cluster_order, ordered=True)
plot_df['gene'] = pd.Categorical(plot_df['gene'], categories=GENES, ordered=True)
plot_df = plot_df.sort_values(['cluster', 'gene'])
plot_df.head(12)

## 4. Plot

In [ ]:
def shorten_label(label):
    return label.replace('neuron_', '') if SHORTEN_CLUSTER_LABELS else label

fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))

if ORIENTATION == 'vertical':
    x_lookup = {gene: i for i, gene in enumerate(GENES)}
    y_lookup = {cluster: i for i, cluster in enumerate(cluster_order)}
elif ORIENTATION == 'horizontal':
    x_lookup = {cluster: i for i, cluster in enumerate(cluster_order)}
    y_lookup = {gene: i for i, gene in enumerate(GENES)}
else:
    raise ValueError("ORIENTATION must be 'vertical' or 'horizontal'")

sc_handle = None
for _, row in plot_df.iterrows():
    if ORIENTATION == 'vertical':
        x, y = x_lookup[str(row['gene'])], y_lookup[str(row['cluster'])]
    else:
        x, y = x_lookup[str(row['cluster'])], y_lookup[str(row['gene'])]
    size = DOT_SIZE_MIN + row['pct_expr'] * DOT_SIZE_MAX_ADD
    sc_handle = ax.scatter(
        x, y, s=size, c=row['mean_scaled_0_1'],
        cmap=cmap_wdg, vmin=0, vmax=1,
        edgecolor='#4A4A4A', linewidth=0.35, zorder=3,
    )

if ORIENTATION == 'vertical':
    ax.set_xticks(range(len(GENES)))
    ax.set_xticklabels(GENES, rotation=X_LABEL_ROTATION, ha='center', fontsize=8.5, fontweight='bold')
    ax.set_yticks(range(len(cluster_order)))
    ax.set_yticklabels([shorten_label(c) for c in cluster_order], fontsize=6)
    ax.set_xlim(-AXIS_PAD, len(GENES) - 1 + AXIS_PAD)
    ax.set_ylim(-AXIS_PAD, len(cluster_order) - 1 + AXIS_PAD)
    ax.invert_yaxis()
else:
    ax.set_xticks(range(len(cluster_order)))
    ax.set_xticklabels([shorten_label(c) for c in cluster_order], rotation=45, ha='right', fontsize=6)
    ax.set_yticks(range(len(GENES)))
    ax.set_yticklabels(GENES, fontsize=8.5, fontweight='bold')
    ax.set_xlim(-AXIS_PAD, len(cluster_order) - 1 + AXIS_PAD)
    ax.set_ylim(-AXIS_PAD, len(GENES) - 1 + AXIS_PAD)
    ax.invert_yaxis()

ax.set_xlabel('')
ax.set_ylabel('')
ax.grid(axis='both', color='#E6E6E6', linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(length=0, pad=3)

cbar = fig.colorbar(sc_handle, ax=ax, fraction=0.045, pad=0.03)
cbar.set_label('Mean expression\nscaled per gene', fontsize=7)
cbar.ax.tick_params(labelsize=6)

plt.tight_layout()

for ext in ['pdf', 'svg', 'png']:
    fig.savefig(f'{OUT_PREFIX}.{ext}', bbox_inches='tight', dpi=300)

plot_df.assign(
    cluster=plot_df['cluster'].astype(str),
    gene=plot_df['gene'].astype(str),
).to_csv(f'{OUT_PREFIX}_values.csv', index=False)
pd.DataFrame({'cluster': cluster_order}).to_csv(f'{OUT_PREFIX}_cluster_order.csv', index=False)

plt.show()
print(f'Saved: {OUT_PREFIX}.pdf / .svg / .png + _values.csv + _cluster_order.csv')